# Local analytics over Open Food Facts with DuckDB

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/toclocoinc/lattice-grid-notebooks/blob/main/open-food-facts-duckdb/open_food_facts_duckdb.ipynb)

Explore **4.8 million food products** on a laptop: DuckDB reads the columns we need straight out of the Open Food Facts Parquet file, the grid opens all of them at once, and you filter by country, brand and Nutri-Score, and watch a chart follow your filters. It closes with one SQL aggregation, the median sugar, salt and energy of the biggest categories, shown as a grid. No API key.

**Data:** [Open Food Facts](https://world.openfoodfacts.org) product database, the [`food.parquet` export](https://huggingface.co/datasets/openfoodfacts/product-database) (about 8 GB, 4.8 million products, 100+ columns). Source: **Open Food Facts contributors**. The database is available under the [Open Database License (ODbL)](https://opendatacommons.org/licenses/odbl/1-0/) and its individual contents under the [Database Contents License (DbCL)](https://opendatacommons.org/licenses/dbcl/1-0/); product images are not used here. Anything you derive from the data and share (including the extract this notebook builds) is shared under the ODbL, with the same attribution. The notebook code is MIT-licensed (see the repo `LICENSE`).

**Size:** the first run downloads the 8 GB file once into a local `cache` folder (a couple of minutes on a fast connection), keeps 9 of its 100+ columns in a 120 MB extract and reuses both afterwards. Set `SAMPLE = True` below for a quick run on the first 100,000 products instead (no big download). The grid then holds the 4.8M-row extract in memory (about 1 GB).

In [58]:
%pip install -q "lattice-grid-jupyter[large]" duckdb

Note: you may need to restart the kernel to use updated packages.


## 1. Read only the columns we need

`SAMPLE` is the switch: `False` uses the whole database, `True` reads just the first 100,000 products straight from the web, which is quick but covers fewer countries and categories.

DuckDB reads the Parquet file column by column, so the other 90+ columns are never touched. The nutrients sit in one list column; a small SQL expression pulls the per-100 g sugar, salt and energy out of it, and values that cannot be real (more than 100 g of sugar in 100 g, more than 900 kcal) become empty. The result is written once as a small Parquet extract.

In [59]:
import os, urllib.request, duckdb
from lattice_grid_jupyter import LatticeGridWidget, LatticeChart

SAMPLE = False          # True: first 100,000 products only, no 8 GB download

URL = "https://huggingface.co/datasets/openfoodfacts/product-database/resolve/main/food.parquet"
CACHE = "cache"
os.makedirs(CACHE, exist_ok=True)
duckdb.sql("SET enable_progress_bar = false")
SRC = URL if SAMPLE else f"{CACHE}/food.parquet"
SLIM = f"{CACHE}/off_products{'_sample' if SAMPLE else ''}.parquet"

if not SAMPLE and not os.path.exists(SRC):
    urllib.request.urlretrieve(URL, SRC + ".part")        # one request, about 8 GB
    os.replace(SRC + ".part", SRC)

def per_100g(name):
    return f"list_filter(nutriments, x -> x.name = '{name}')[1].\"100g\""

if not os.path.exists(SLIM):
    if SAMPLE:
        duckdb.sql("INSTALL httpfs; LOAD httpfs")
    duckdb.sql(f"""
        COPY (
          SELECT code, product_name[1].text AS product_name,
                 nullif(trim(split_part(brands, ',', 1)), '') AS brand,
                 replace(countries_tags[1], 'en:', '') AS country,
                 nullif(replace(categories_tags[1], 'en:', ''), 'undefined') AS category,
                 CASE WHEN nutriscore_grade IN ('a','b','c','d','e') THEN upper(nutriscore_grade) END AS nutri_score,
                 CASE WHEN {per_100g('sugars')} BETWEEN 0 AND 100 THEN round({per_100g('sugars')}::DOUBLE, 1) END::DOUBLE AS sugar_g,
                 CASE WHEN {per_100g('salt')} BETWEEN 0 AND 100 THEN round({per_100g('salt')}::DOUBLE, 2) END::DOUBLE AS salt_g,
                 CASE WHEN {per_100g('energy-kcal')} BETWEEN 0 AND 900 THEN round({per_100g('energy-kcal')}::DOUBLE, 0) END::DOUBLE AS energy_kcal,
                 1 AS products
          FROM read_parquet('{SRC}') {"LIMIT 100000" if SAMPLE else ""}
        ) TO '{SLIM}' (FORMAT parquet, COMPRESSION zstd)
    """)

duckdb.sql(f"""
    SELECT count(*) AS products, count(nutri_score) AS with_nutri_score, count(sugar_g) AS with_sugar,
           count(DISTINCT country) AS countries, count(DISTINCT category) AS categories,
           count(DISTINCT brand) AS brands
    FROM '{SLIM}'
""").df()

,products,with_nutri_score,with_sugar,countries,categories,brands
0,4826073,1424011,3243666,314,39983,476098


## 2. Open every product in a grid

The grid is given the extract's Parquet file, so the products never pass through pandas. Above 100,000 rows the widget uses its **large-dataframe mode**: it sends no rows up front, the grid asks Python for each window as you scroll, and sort, filter, the header histograms and the Statistics tab are computed over every row. (In this mode the widget holds the extract in memory and answers the grid's queries itself; DuckDB did the reading and shaping.)

It opens filtered to **French products with a Nutri-Score** (the `country` and `nutri_score` filters). The histograms in the nutrient headers redraw for the filtered products. Change the country, pick a brand in the `brand` filter menu, or click a Nutri-Score letter and watch the chart under the grid follow. The chart counts the filtered products per Nutri-Score grade.

In [60]:
from ipywidgets import VBox
from IPython.display import clear_output

widths = dict(code=130, product_name=270, brand=150, country=130, category=210, nutri_score=120,
              sugar_g=90, salt_g=90, energy_kcal=110)
columns = [{"field": f, "layout": {"width": x}} for f, x in widths.items()]
columns.append({"field": "products", "layout": {"hidden": True}})
france_scored = {"op": "and", "conditions": [
    {"col": "country", "op": "eq", "value": "france"},
    {"col": "nutri_score", "op": "notBlank"}]}

grid = LatticeGridWidget(SLIM, histograms=True, profile=True, height=480, options={"header_height": 90},
                         columns=columns, state={"filters": france_scored})
chart = LatticeChart(grid, type="bar", x="nutri_score", y="products", aggregate="sum", height=300)
clear_output()          # DuckDB's own loading bar, left over from reading the extract
n_products = duckdb.sql(f"SELECT count(*) FROM '{SLIM}'").fetchone()[0]
print(f"{n_products:,} products in the grid; large-dataframe mode: {grid.windowed}")
VBox([grid, chart])

4,826,073 products in the grid; large-dataframe mode: True


![the grid](https://raw.githubusercontent.com/toclocoinc/lattice-grid-notebooks/main/open-food-facts-duckdb/grid.png)

*The grid in JupyterLab: a histogram strip in every header, the filter on country and Nutri-Score, and the Nutri-Score chart bound to the same grid.*

## 3. How many products match?

`grid.view` is the filtered, sorted rows (a lazy handle in this mode): its length is the number of matching products, and `.head(n)` pulls the first `n` of them into a pandas `DataFrame`.

In [61]:
print(f"{len(grid.view):,} products match the grid's filter")
grid.view.head(5).drop(columns="products")

451,186 products match the grid's filter


,code,product_name,brand,country,category,nutri_score,sugar_g,salt_g,energy_kcal
0,0019000025005,Life Savers Hard Candy Sugar Free,Life Saver,france,snacks,C,0.0,0.00,200.0
1,0019000082244,"Fruit variety hard candy, watermelon, pineappl...",Lifesavers,france,snacks,E,80.0,0.00,400.0
2,0019281700110,Boisson a base de jus de Lychee,Tang freres,france,beverages-and-beverages-preparations,E,10.0,0.02,44.0
3,0019281700127,Boisson à base de jus de goyave,Tang frères,france,beverages-and-beverages-preparations,E,9.9,0.10,41.0
4,0019281700134,Boisson à base d annanas,Cock Brand,france,beverages-and-beverages-preparations,E,12.0,0.00,50.0


## 4. Group by category: one SQL aggregation, shown as a grid

The same extract answers a grouping question in plain SQL: for each of the 15 biggest categories (the first category tag of each product), how many products there are, how many have a Nutri-Score, the share graded A or B, and the **median** sugar, salt and energy per 100 g. `.df()` turns that small result into a DataFrame for a second grid.

In [62]:
by_category = duckdb.sql(f"""
    SELECT category, count(*) AS products,
           count(nutri_score) AS scored,
           round(100.0 * count(*) FILTER (nutri_score IN ('A', 'B')) / count(nutri_score), 1) AS pct_a_or_b,
           round(median(sugar_g)::DOUBLE, 1) AS median_sugar_g,
           round(median(salt_g)::DOUBLE, 2) AS median_salt_g,
           round(median(energy_kcal)::DOUBLE, 0) AS median_energy_kcal
    FROM '{SLIM}' WHERE category IS NOT NULL
    GROUP BY category HAVING count(nutri_score) > 0 ORDER BY products DESC LIMIT 15
""").df()
LatticeGridWidget(by_category, height=470, histograms=True, options={"header_height": 90},
                  columns=[{"field": f, "layout": {"width": x}} for f, x in
                           dict(category=250, products=100, scored=100, pct_a_or_b=110, median_sugar_g=140,
                                median_salt_g=130, median_energy_kcal=170).items()])

---
Data: Open Food Facts contributors, [ODbL](https://opendatacommons.org/licenses/odbl/1-0/) / [DbCL](https://opendatacommons.org/licenses/dbcl/1-0/), from the [Open Food Facts product database](https://huggingface.co/datasets/openfoodfacts/product-database). Shared derivatives stay under the ODbL.